# Tests et Validation du Dataset DataCenter

Ce notebook inspecte, valide et prépare le dataset d'entraînement pour le fine-tuning LoRA.

**Objectifs:**
- Charger et explorer les données brutes
- Valider la structure (instruction/input/output)
- Détecter doublons, anomalies et outliers
- Générer statistiques sur les tokens
- Préparer dataset formaté pour l'entraînement

## 1. Setup et chargement des données

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

# Chemins
DATA_RAW = Path('/vercel/share/v0-project/data/raw')
DATA_PROCESSED = Path('/vercel/share/v0-project/data/processed')
DATA_PROCESSED.mkdir(exist_ok=True)

print(f"Raw data path: {DATA_RAW}")
print(f"Processed data path: {DATA_PROCESSED}")
print(f"Files in data/raw: {list(DATA_RAW.glob('*'))}")

## 2. Charger dataset brut

In [ ]:
# Charger le dataset JSONL
dataset_file = DATA_RAW / 'datacenter_qa.jsonl'

data = []
with open(dataset_file, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        try:
            record = json.loads(line)
            record['line_num'] = line_num
            data.append(record)
        except json.JSONDecodeError as e:
            print(f"Erreur ligne {line_num}: {e}")

df = pd.DataFrame(data)
print(f"\nDataset chargé: {len(df)} exemples")
print(f"\nColonnes: {df.columns.tolist()}")
print(f"\nFormat:")
print(df.head(3).to_string())

## 3. Validation et nettoyage

In [ ]:
# Vérifier les colonnes requises
required_cols = ['instruction', 'input', 'output']
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"ERREUR: Colonnes manquantes: {missing_cols}")
else:
    print("✓ Toutes les colonnes requises présentes")

# Vérifier les valeurs nulles
print("\nVérification des valeurs nulles:")
null_counts = df[required_cols].isnull().sum()
print(null_counts)

if null_counts.sum() > 0:
    print(f"\nAttention: {null_counts.sum()} valeurs nulles détectées")
    df_clean = df.dropna(subset=required_cols)
    print(f"Après suppression: {len(df_clean)} exemples")
else:
    df_clean = df
    print("✓ Aucune valeur nulle")

# Vérifier les doublons
duplicates = df_clean.duplicated(subset=['instruction', 'output'])
print(f"\nDoublons: {duplicates.sum()} trouvés")
if duplicates.sum() > 0:
    print("Exemples de doublons:")
    print(df_clean[duplicates].head())
    df_clean = df_clean[~duplicates]
    print(f"Après suppression des doublons: {len(df_clean)} exemples")

## 4. Analyse statistique des textes

In [ ]:
# Statistiques de longueur
df_clean['instruction_len'] = df_clean['instruction'].str.len()
df_clean['input_len'] = df_clean['input'].str.len()
df_clean['output_len'] = df_clean['output'].str.len()
df_clean['total_len'] = df_clean['instruction_len'] + df_clean['input_len'] + df_clean['output_len']

print("\n" + "="*70)
print("STATISTIQUES DE LONGUEUR (caractères)")
print("="*70)

stats_df = pd.DataFrame({
    'Min': df_clean[['instruction_len', 'input_len', 'output_len', 'total_len']].min(),
    'Max': df_clean[['instruction_len', 'input_len', 'output_len', 'total_len']].max(),
    'Mean': df_clean[['instruction_len', 'input_len', 'output_len', 'total_len']].mean(),
    'Std': df_clean[['instruction_len', 'input_len', 'output_len', 'total_len']].std(),
    'Median': df_clean[['instruction_len', 'input_len', 'output_len', 'total_len']].median()
})

print(stats_df.round(2).to_string())

# Déterminer les outliers
Q1 = df_clean['total_len'].quantile(0.25)
Q3 = df_clean['total_len'].quantile(0.75)
IQR = Q3 - Q1
outliers = df_clean[(df_clean['total_len'] < Q1 - 1.5*IQR) | (df_clean['total_len'] > Q3 + 1.5*IQR)]

print(f"\nOutliers détectés (par IQR): {len(outliers)}")
if len(outliers) > 0:
    print("Exemples d'outliers:")
    print(outliers[['instruction', 'output_len']].head().to_string())

## 5. Analyse des tokens avec le tokenizer Qwen

In [ ]:
# Charger le tokenizer Qwen
print("Chargement du tokenizer Qwen...")
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')

# Compter les tokens
def count_tokens(text):
    """Compte le nombre de tokens"""
    return len(tokenizer.encode(text))

df_clean['instruction_tokens'] = df_clean['instruction'].apply(count_tokens)
df_clean['input_tokens'] = df_clean['input'].apply(count_tokens)
df_clean['output_tokens'] = df_clean['output'].apply(count_tokens)
df_clean['total_tokens'] = df_clean['instruction_tokens'] + df_clean['input_tokens'] + df_clean['output_tokens']

print("\n" + "="*70)
print("STATISTIQUES DE TOKENS")
print("="*70)

token_stats = pd.DataFrame({
    'Min': df_clean[['instruction_tokens', 'input_tokens', 'output_tokens', 'total_tokens']].min(),
    'Max': df_clean[['instruction_tokens', 'input_tokens', 'output_tokens', 'total_tokens']].max(),
    'Mean': df_clean[['instruction_tokens', 'input_tokens', 'output_tokens', 'total_tokens']].mean(),
    'Median': df_clean[['instruction_tokens', 'input_tokens', 'output_tokens', 'total_tokens']].median()
})

print(token_stats.round(2).to_string())

# Distribution des tokens
print(f"\nGran Max tokens (10%): {df_clean['total_tokens'].quantile(0.9):.0f}")
print(f"Seuil 512 tokens: {(df_clean['total_tokens'] <= 512).sum()} / {len(df_clean)} ({(df_clean['total_tokens'] <= 512).sum()/len(df_clean)*100:.1f}%)")
print(f"Seuil 1024 tokens: {(df_clean['total_tokens'] <= 1024).sum()} / {len(df_clean)} ({(df_clean['total_tokens'] <= 1024).sum()/len(df_clean)*100:.1f}%)")

## 6. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Analyse du Dataset DataCenter', fontsize=16, fontweight='bold')

# Distribution des longueurs de réponse
ax = axes[0, 0]
ax.hist(df_clean['output_len'], bins=20, color='skyblue', edgecolor='black')
ax.set_title('Distribution de la longueur des réponses')
ax.set_xlabel('Longueur (caractères)')
ax.set_ylabel('Fréquence')
ax.axvline(df_clean['output_len'].mean(), color='red', linestyle='--', label=f"Moyenne: {df_clean['output_len'].mean():.0f}")
ax.legend()

# Distribution des tokens total
ax = axes[0, 1]
ax.hist(df_clean['total_tokens'], bins=20, color='lightgreen', edgecolor='black')
ax.set_title('Distribution du total de tokens')
ax.set_xlabel('Tokens')
ax.set_ylabel('Fréquence')
ax.axvline(df_clean['total_tokens'].mean(), color='red', linestyle='--', label=f"Moyenne: {df_clean['total_tokens'].mean():.0f}")
ax.legend()

# Répartition instruction/input/output en tokens
ax = axes[1, 0]
token_breakdown = [
    df_clean['instruction_tokens'].sum(),
    df_clean['input_tokens'].sum(),
    df_clean['output_tokens'].sum()
]
ax.pie(token_breakdown, labels=['Instruction', 'Input', 'Output'], autopct='%1.1f%%', colors=['#ff9999', '#66b3ff', '#99ff99'])
ax.set_title('Répartition des tokens par type')

# Scatter plot: instruction vs output
ax = axes[1, 1]
ax.scatter(df_clean['instruction_tokens'], df_clean['output_tokens'], alpha=0.6, s=50)
ax.set_title('Relation Instruction vs Output')
ax.set_xlabel('Tokens Instruction')
ax.set_ylabel('Tokens Output')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'dataset_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("Graphiques sauvegardés: data/processed/dataset_analysis.png")

## 7. Détails des catégories (domaines DataCenter)

In [ ]:
# Extraire les catégories depuis les instructions
def extract_category(instruction):
    """Extrait la catégorie générale de l'instruction"""
    keywords = {
        'monitoring': ['monitor', 'check', 'alert', 'metric', 'dashboard'],
        'security': ['security', 'ssh', 'ssl', 'certificate', 'firewall', 'patch'],
        'storage': ['disk', 'storage', 'backup', 'raid'],
        'networking': ['network', 'bandwidth', 'connection', 'latency'],
        'performance': ['performance', 'optimize', 'cpu', 'memory', 'temperature'],
        'maintenance': ['shutdown', 'decommission', 'deployment', 'upgrade'],
        'database': ['database', 'query', 'replication'],
        'infrastructure': ['redundancy', 'failover', 'load']
    }
    
    instruction_lower = instruction.lower()
    for category, keywords_list in keywords.items():
        if any(kw in instruction_lower for kw in keywords_list):
            return category
    return 'other'

df_clean['category'] = df_clean['instruction'].apply(extract_category)

print("\n" + "="*70)
print("DISTRIBUTION PAR CATÉGORIE")
print("="*70)

category_stats = df_clean.groupby('category').agg({
    'instruction': 'count',
    'total_tokens': ['mean', 'max']
}).round(2)

category_stats.columns = ['Count', 'Avg Tokens', 'Max Tokens']
category_stats = category_stats.sort_values('Count', ascending=False)
print(category_stats.to_string())

# Visualiser la distribution des catégories
fig, ax = plt.subplots(figsize=(10, 6))
category_counts = df_clean['category'].value_counts()
ax.barh(category_counts.index, category_counts.values, color='coral')
ax.set_title('Distribution des Exemples par Catégorie')
ax.set_xlabel('Nombre d\'exemples')

# Ajouter les valeurs sur les barres
for i, v in enumerate(category_counts.values):
    ax.text(v + 0.1, i, str(v), va='center')

plt.tight_layout()
plt.savefig(DATA_PROCESSED / 'category_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nGraphique sauvegardé: data/processed/category_distribution.png")

## 8. Préparation du dataset final pour training

In [ ]:
# Format final pour LoRA: texte combiné
def format_for_training(row):
    """Formate un exemple pour le fine-tuning LoRA"""
    instruction = row['instruction'].strip()
    input_text = row['input'].strip()
    output = row['output'].strip()
    
    # Format Alpaca-style
    if input_text:
        text = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output}"""
    else:
        text = f"""### Instruction:
{instruction}

### Response:
{output}"""
    
    return text

# Créer le dataset formaté
df_clean['formatted_text'] = df_clean.apply(format_for_training, axis=1)

# Sauvegarder en format texte
train_file = DATA_PROCESSED / 'train.jsonl'

with open(train_file, 'w', encoding='utf-8') as f:
    for _, row in df_clean.iterrows():
        record = {
            'text': row['formatted_text'],
            'instruction': row['instruction'],
            'category': row['category']
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f"\n✓ Dataset sauvegardé: {train_file}")
print(f"  Total d'exemples: {len(df_clean)}")

# Afficher un exemple
print("\nExemple de formatage (1er exemple):")
print("="*70)
print(df_clean['formatted_text'].iloc[0])
print("="*70)

## 9. Rapport final de qualité

In [ ]:
print("\n" + "="*70)
print("RAPPORT FINAL DE QUALITÉ DU DATASET")
print("="*70)

print(f"""
STATISTIQUES GLOBALES:
  - Nombre d'exemples: {len(df_clean)}
  - Exemples supprimés (doublons/nulls): {len(df) - len(df_clean)}
  - Taux de rétention: {len(df_clean)/len(df)*100:.1f}%

TOKENS:
  - Total tokens: {df_clean['total_tokens'].sum():,.0f}
  - Moyenne par exemple: {df_clean['total_tokens'].mean():.1f}
  - Max par exemple: {df_clean['total_tokens'].max():.0f}
  - Exemples > 512 tokens: {(df_clean['total_tokens'] > 512).sum()} ({(df_clean['total_tokens'] > 512).sum()/len(df_clean)*100:.1f}%)

DISTRIBUTION PAR CATÉGORIE:
""")

for cat, count in df_clean['category'].value_counts().items():
    print(f"  - {cat}: {count} ({count/len(df_clean)*100:.1f}%)")

print(f"""
QUALITÉ:
  - Doublons: 0 ✓
  - Valeurs nulles: 0 ✓
  - Formato cohérent: Oui ✓

FICHIERS GÉNÉRÉS:
  - {train_file}
  - {DATA_PROCESSED / 'dataset_analysis.png'}
  - {DATA_PROCESSED / 'category_distribution.png'}

PRÊT POUR LE FINE-TUNING: ✓ OUI

Prochaine étape: Lancer le fine-tuning LoRA
  → Voir notebook 03_demo_finetuning.ipynb
""")